In [1]:
from pymongo import MongoClient
from pprint import pprint

client = MongoClient("mongodb://localhost:27017/")
db = client["intern_db"]

events = db["events"]
products = db["products"]
customers = db["customers"]

print("Connected to MongoDB")

Connected to MongoDB


In [2]:
print("Events indexes:")
pprint(list(events.list_indexes()))

print("\nProducts indexes:")
pprint(list(products.list_indexes()))

print("\nCustomers indexes:")
pprint(list(customers.list_indexes()))

Events indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])]

Products indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])]

Customers indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])]


In [3]:
query = {"event_type": "purchase"}

explain_before_index = events.find(query).explain()

pprint(explain_before_index["executionStats"])

{'allPlansExecution': [],
 'executionStages': {'advanced': 1,
                     'direction': 'forward',
                     'docsExamined': 3,
                     'executionTimeMillisEstimate': 0,
                     'filter': {'event_type': {'$eq': 'purchase'}},
                     'isCached': False,
                     'isEOF': 1,
                     'nReturned': 1,
                     'needTime': 2,
                     'needYield': 0,
                     'nss': 'intern_db.events',
                     'restoreState': 0,
                     'saveState': 0,
                     'stage': 'COLLSCAN',
                     'works': 4},
 'executionSuccess': True,
 'executionTimeMillis': 44,
 'nReturned': 1,
 'totalDocsExamined': 3,
 'totalKeysExamined': 0}


In [4]:
events.create_index("event_type")
events.create_index("product_id")
events.create_index("customer_id")

products.create_index("product_id")
customers.create_index("customer_id")

print("Indexes created successfully.")

Indexes created successfully.


In [5]:
query = {"event_type": "purchase"}

explain_after_index = events.find(query).explain()

pprint(explain_after_index["executionStats"])

{'allPlansExecution': [],
 'executionStages': {'advanced': 1,
                     'alreadyHasObj': 0,
                     'docsExamined': 1,
                     'executionTimeMillisEstimate': 21,
                     'inputStage': {'advanced': 1,
                                    'direction': 'forward',
                                    'dupsDropped': 0,
                                    'dupsTested': 0,
                                    'executionTimeMillisEstimate': 21,
                                    'indexBounds': {'event_type': ['["purchase", '
                                                                   '"purchase"]']},
                                    'indexName': 'event_type_1',
                                    'indexVersion': 2,
                                    'isEOF': 1,
                                    'isMultiKey': False,
                                    'isPartial': False,
                                    'isSparse': False,
         

In [6]:
print("Events indexes:")
pprint(list(events.list_indexes()))

print("\nProducts indexes:")
pprint(list(products.list_indexes()))

print("\nCustomers indexes:")
pprint(list(customers.list_indexes()))

Events indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('event_type', 1)])), ('name', 'event_type_1')]),
 SON([('v', 2), ('key', SON([('product_id', 1)])), ('name', 'product_id_1')]),
 SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'customer_id_1')])]

Products indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('product_id', 1)])), ('name', 'product_id_1')])]

Customers indexes:
[SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')]),
 SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'customer_id_1')])]


In [7]:
events.create_index([
    ("event_type", 1),
    ("product_id", 1)
])

print("Compound index created successfully.")

Compound index created successfully.


In [8]:
compound_query = {
    "event_type": "purchase",
    "product_id": 1
}

compound_explain = events.find(compound_query).explain()

pprint(compound_explain["executionStats"])

{'allPlansExecution': [{'executionStages': {'advanced': 1,
                                            'alreadyHasObj': 0,
                                            'docsExamined': 1,
                                            'executionTimeMillisEstimate': 22,
                                            'inputStage': {'advanced': 1,
                                                           'direction': 'forward',
                                                           'dupsDropped': 0,
                                                           'dupsTested': 0,
                                                           'executionTimeMillisEstimate': 22,
                                                           'indexBounds': {'event_type': ['["purchase", '
                                                                                          '"purchase"]'],
                                                                           'product_id': ['[1, '
                      

In [9]:
products.create_index([
    ("product_name", "text"),
    ("category", "text")
])

print("Text index created successfully.")

Text index created successfully.


In [10]:
text_results = products.find(
    {
        "$text": {
            "$search": "Electronics"
        }
    },
    {
        "_id": 0,
        "product_name": 1,
        "category": 1,
        "price": 1
    }
)

for product in text_results:
    pprint(product)

{'category': 'Electronics', 'price': 350, 'product_name': 'Headphones'}
{'category': 'Electronics', 'price': 3200, 'product_name': 'Laptop'}
